# ISRO AQI — CNN-LSTM Training (Google Colab T4)
**Instructions:**
1. Upload `data/processed/aqi_cube.zarr` and `data/processed/cpcb_stations.parquet` to Google Drive under `MyDrive/ISRO/`
2. Upload `ml/` and `etl/` folders to Google Drive under `MyDrive/ISRO/code/`
3. Set Runtime → Change runtime type → **T4 GPU**
4. Run all cells

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q xarray zarr pyarrow scikit-learn tqdm

In [ ]:
import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/ISRO'
CODE_ROOT  = f'{DRIVE_ROOT}/code'
sys.path.insert(0, CODE_ROOT)

# Symlink data to expected path
os.makedirs('/content/data/processed', exist_ok=True)
os.makedirs('/content/data/serving/models', exist_ok=True)

!ln -sfn {DRIVE_ROOT}/aqi_cube.zarr          /content/data/processed/aqi_cube.zarr
!ln -sfn {DRIVE_ROOT}/cpcb_stations.parquet  /content/data/processed/cpcb_stations.parquet

# Patch config paths
import etl.config as cfg
from pathlib import Path
cfg.ZARR_AQI      = Path('/content/data/processed/aqi_cube.zarr')
cfg.CPCB_PARQUET  = Path('/content/data/processed/cpcb_stations.parquet')
cfg.MODELS_DIR    = Path('/content/data/serving/models')
print('Paths configured')

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
from ml.obj1_aqi.train import train

model, history = train(
    epochs=25,
    batch_size=128,
    lr=1e-3,
    device='cuda',
)

In [ ]:
# Plot training history
import matplotlib.pyplot as plt
import json

with open('/content/data/serving/models/train_history.json') as f:
    history = json.load(f)

epochs   = [h['epoch']     for h in history]
val_rmse = [h['val_rmse']  for h in history]
train_l  = [h['train_loss'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, train_l);   axes[0].set_title('Train Loss (Huber)')
axes[1].plot(epochs, val_rmse);  axes[1].set_title('Val RMSE (AQI)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ISRO/training_curves.png', dpi=150)
plt.show()
print(f'Best val RMSE: {min(val_rmse):.2f}')

In [ ]:
# Copy model to Drive
import shutil
shutil.copy('/content/data/serving/models/cnn_lstm_best.pt', f'{DRIVE_ROOT}/cnn_lstm_best.pt')
shutil.copy('/content/data/serving/models/scaler.pkl',       f'{DRIVE_ROOT}/scaler.pkl')
print('Model saved to Drive. Download to data/serving/models/ locally.')

In [ ]:
# Run batch prediction on full grid
from ml.obj1_aqi.predict import predict_grid
predict_grid(n_passes=10)

# Copy predictions to Drive
shutil.copytree('/content/data/processed/aqi_predictions.zarr',
                f'{DRIVE_ROOT}/aqi_predictions.zarr', dirs_exist_ok=True)
print('Predictions saved to Drive.')